In [7]:
import re
import pandas as pd
import os 

notebook_path= os.getcwd()
in_dir_notifications  = os.path.abspath(os.path.join(notebook_path,"..","..","Data","notification_historical","updated_notifications.jsonl"))
in_dir_master_direction = os.path.abspath(os.path.join(notebook_path,"..","..","Data","master_directory","master_directory.jsonl"))
circulars = pd.read_json(in_dir_notifications, lines=True)
master_directions = pd.read_json(in_dir_master_direction, lines = True)
# regexes to catch "Reserve Bank of India (X) ... Directions" and the no-paren variant
p1 = re.compile(r"Reserve Bank of India\s*[-–—]?\s*\(([^)]+)\)\s*(?:[A-Za-z]+\s+){0,3}Directions", re.IGNORECASE)
p2 = re.compile(r"Reserve Bank of India\s*[-–—]\s*([A-Za-z ,]+?),?\s*Directions", re.IGNORECASE)

norm = lambda s: re.sub(r"\s+", " ", s.replace("–","-").replace("—","-")).strip(" ,-").lower()

# extract candidate names from title + text
circulars["found_title"] = circulars["title"].apply(lambda t: [norm(x) for x in (p1.findall(t) + p2.findall(t))] if isinstance(t, str) else [])
circulars["found_text"]  = circulars["text"].apply(lambda t: [norm(x) for x in (p1.findall(t) + p2.findall(t))] if isinstance(t, str) else [])

# build master lookup: normalized name -> master direction id
master_lookup = {}
for _, r in master_directions.iterrows():
    for name in [norm(x) for x in (p1.findall(r["title"]) + p2.findall(r["title"]))]:
        master_lookup[name] = r["id"]

# match: prefer title hit, fall back to text hit
def _match(row):
    for n in row["found_title"]:
        if n in master_lookup: return master_lookup[n], "title_match"
    for n in row["found_text"]:
        if n in master_lookup: return master_lookup[n], "text_match"
    return None, "no_match",

circulars[["matched_id", "match_method"]] = circulars.apply(lambda r: pd.Series(_match(r)), axis=1)

# stats
print(circulars["match_method"].value_counts())
print(f"matched: {(circulars['match_method']!='no_match').mean():.1%}")

circulars[["id","title","found_title","matched_id","match_method"]]

match_method
no_match       97
title_match    34
text_match      7
Name: count, dtype: int64
matched: 29.7%


,id,title,found_title,matched_id,match_method
0,13167,Reserve Bank of India (Setting Up of Wholly Ow...,[],NaN,no_match
1,13168,Reserve Bank of India (Universal Banks – Licen...,[],NaN,no_match
2,13169,Compliance with Know Your Customer (KYC) norms,[],12943.0,text_match
3,13170,Consolidation of Regulations – Withdrawal of c...,[],NaN,no_match
4,13171,Compliance with Know Your Customer (KYC) norms,[],NaN,no_match
...,...,...,...,...,...
133,13675,Formation of new districts in the Union Territ...,[],NaN,no_match
134,13676,"Implementation of Section 51A of UAPA, 1967: U...",[],NaN,no_match
135,13677,"Implementation of Section 51A of UAPA, 1967: U...",[],NaN,no_match
136,13678,"Implementation of Section 51A of UAPA, 1967: U...",[],NaN,no_match


In [10]:
no_match = circulars[circulars["match_method"] == "no_match"]
print(len(no_match))
for t in no_match["title"].head(10):
    print("-", t)
no_match["text"].iloc[0]

97
- Reserve Bank of India (Setting Up of Wholly Owned Subsidiaries by Foreign Banks) Guidelines, 2025 (Updated as on April 1, 2026)
- Reserve Bank of India (Universal Banks – Licensing) Guidelines, 2025
- Consolidation of Regulations – Withdrawal of circulars
- Compliance with Know Your Customer (KYC) norms
- Liberalised Remittance Scheme (LRS)- Submission of ‘LRS Daily Return’ by Authorised Dealers- Category -II banks/ entities and Full- Fledged Money Changers
- Liquidity Adjustment Facility - Change in rates
- Standing Liquidity Facility for Primary Dealers
- Penal Interest on shortfall in CRR and SLR requirements - Change in Bank Rate
- Reserve Bank of India (Non-Operative Financial Holding Company) (Amendment) Directions, 2025
- Export and Import of Indian Currency to or from Nepal and Bhutan


'RBI/DOR/2025-26/144\nNovember 28, 2025\nPrevious Versions\nReserve Bank of India (Setting Up of Wholly Owned Subsidiaries by Foreign Banks) Guidelines, 2025 (Updated as on April 1, 2026)\nA. Background\nThe global financial crisis of 2008 demonstrated that the growing complexity and interconnectedness of financial institutions, coupled with the lack of effective cross-border resolution regimes, severely constrained the ability of home and host authorities to cope with the failure of too big to fail (TBTF) and too connected to fail (TCTF) institutions. Globally, several policy options have been proposed to address these challenges, including measures to contain the negative externalities arising out of size and interconnectedness, strengthening the capital and liquidity buffers, and enhancing the resolvability of such institutions. The lessons from the global financial crisis support the case for domestic incorporation of foreign banks. The main advantages of local incorporation includ